# Diffusion Models: от шумового расписания до нормальной статистики $Z_t$

Этот ноутбук — пошаговое построение диффузионной модели с нуля.  
Цель не генерация, а получение **скалярной нормированной статистики $Z_t$**, которая:
- распределена как $\mathcal{N}(0,1)$ на нормальных данных (верифицируем),
- смещается при изменении распределения.

Блоки идут от простого к сложному и логически зависят друг от друга:

| Блок | Содержание |
|------|------------|
| 1 | Шумовое расписание VP-SDE |
| 2 | Прямой процесс: добавление шума |
| 3 | DSM-лосс: чему учим модель |
| 4 | Архитектура деноизера |
| 5 | Цикл обучения |
| 6 | Sanity checks: score на гауссовских данных |
| 7 | Статистика $L_t$: $K$ независимых шумов |
| 8 | Калибровка $\mu_0$, $\sigma_0$ и нормировка в $Z_t$ |
| 9 | Проверка нормальности $Z_t$ |
| 10 | Демонстрация сдвига под $p_1 \neq p_0$ |

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import scipy.stats as stats
from torch.utils.data import DataLoader
import math

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

In [ ]:
math.log(2.7)

---
## Блок 1. Шумовое расписание VP-SDE

### Прямой процесс

Рассматриваем **VP-SDE** (Variance Preserving):

$$dx_\tau = -\frac{\beta(\tau)}{2}\, x_\tau\, d\tau + \sqrt{\beta(\tau)}\, dW_\tau, \qquad \tau \in [0, T]$$

Линейный дрейф + гауссовский шум $\Rightarrow$ переходное ядро тоже гауссово:

$$p(x_\tau \mid x_0) = \mathcal{N}\!\left(x_\tau;\; \mu(\tau)\, x_0,\; \sigma^2(\tau)\, I\right)$$

где масштабирующие функции:

$$\mu(\tau) = \exp\!\left(-\frac{1}{2}\int_0^\tau \beta(s)\, ds\right), \qquad \sigma^2(\tau) = 1 - \mu^2(\tau)$$

### Линейное расписание

Простейший выбор: $\beta(\tau) = \beta_{\min} + (\beta_{\max} - \beta_{\min})\cdot\tau/T$.

Тогда интеграл аналитически:

$$\int_0^\tau \beta(s)\, ds = \beta_{\min}\tau + \frac{(\beta_{\max} - \beta_{\min})\tau^2}{2T}$$

### Дискретная версия (для обучения)

На практике $\tau \in \{1, \ldots, T\}$ дискретный (DDPM-style). Шаги:

$$\bar\alpha_t = \prod_{s=1}^t (1 - \beta_s), \qquad \mu(t) = \sqrt{\bar\alpha_t}, \qquad \sigma(t) = \sqrt{1 - \bar\alpha_t}$$

Свойства: $\mu(0) = 1$, $\sigma(0) = 0$ (нет шума); $\mu(T) \approx 0$, $\sigma(T) \approx 1$ (чистый гауссовский шум).

In [ ]:
class NoiseSchedule:
    """
    Линейное шумовое расписание DDPM.

    Параметры
    ---------
    T        : число дискретных шагов
    beta_min : минимальное значение β
    beta_max : максимальное значение β
    """

    def __init__(self, T: int = 1000, beta_min: float = 1e-4, beta_max: float = 0.02):
        self.T = T

        self.betas = torch.linspace(beta_min, beta_max, T) # beta_t = linspace(beta_min, beta_max, T) — тензор длины T
        self.alphas = 1 - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, 0) # alpha_bar_t = cumprod(alpha_t)
        self.mus    = torch.sqrt(self.alpha_bars)# sqrt(alpha_bar[t])
        self.sigmas = torch.sqrt((1 - self.alpha_bars))# sqrt(1 - alpha_bar[t])

    def get(self, t: torch.Tensor):
        """
        Вернуть (mu, sigma) для батча временных шагов t.

        Параметры
        ---------
        t : torch.LongTensor, shape (B,) — индексы от 0 до T-1

        Возвращает
        ----------
        mu    : shape (B, 1, ..., 1) — broadcastable с данными
        sigma : shape (B, 1, ..., 1)
        """
        if len(t.shape) > 1:
            return self.mus[t], self.sigmas[t]
        else:
            return self.mus[t, None], self.sigmas[t, None]

# --- Проверка: визуализируем расписание ---
schedule = NoiseSchedule(T=1000)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
t_range = np.arange(1000)

# TODO: построить графики beta_t, mu(t), sigma(t) по t
# axes[0]: beta_t
# axes[1]: mu(t) и sigma(t)
# axes[2]: mu^2 + sigma^2 (должно быть = 1 — проверка VP-свойства)

axes[0].plot(schedule.betas)
axes[0].set_title('β(t)')
axes[1].plot(schedule.mus, label='μ(t)')
axes[1].plot(schedule.sigmas, label='σ(t)')
axes[1].legend()
axes[1].set_title('μ(t) и σ(t)')
axes[2].plot(schedule.mus**2 + schedule.sigmas**2)
axes[2].axhline(1, color='r', linestyle='--', alpha=0.5)
axes[2].set_title('μ²+σ² (должно быть = 1)')
axes[2].set_ylim(0, 1.2)

plt.tight_layout()
plt.show()

---
## Блок 2. Прямой процесс: добавление шума

### Замкнутая формула (ключевое свойство VP-SDE)

Прямой процесс можно прогнать **за один шаг** из $x_0$ напрямую в $x_\tau$:

$$x_\tau = \mu(\tau)\, x_0 + \sigma(\tau)\, \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I)$$

Это следует из того, что $p(x_\tau \mid x_0)$ — гауссово с параметрами $(\mu(\tau) x_0,\; \sigma^2(\tau) I)$.

### Что мы сохраняем

При обучении сохраняем три вещи:
- зашумлённый вход $x_\tau$ (подаём в модель),
- шаг $\tau$ (подаём в модель),
- добавленный шум $\varepsilon$ (**цель** предсказания).

### Зачем нужно VP-свойство

$$\mathbb{E}[\|x_\tau\|^2] = \mu^2(\tau)\,\mathbb{E}[\|x_0\|^2] + \sigma^2(\tau)\, d \approx \mathbb{E}[\|x_0\|^2]$$

при $\mu^2 + \sigma^2 = 1$. Дисперсия не взрывается и не коллапсирует — это важно для стабильного обучения в высоких размерностях.

In [ ]:
def forward_process(x0: torch.Tensor, t: torch.Tensor, schedule: NoiseSchedule):
    """
    Прямой процесс: зашумить x0 до уровня t.

    Параметры
    ---------
    x0       : (B, d) — оригинальные данные
    t        : (B,)   — шаги шума, целые от 0 до T-1
    schedule : NoiseSchedule

    Возвращает
    ----------
    x_t   : (B, d) — зашумлённые данные
    eps   : (B, d) — добавленный шум (цель предсказания)
    mu    : (B, 1) — коэффициент при x0
    sigma : (B, 1) — стандартное отклонение шума
    """
    # TODO:
    # 1. Сэмплировать eps ~ N(0, I) того же размера, что x0
    # 2. Достать mu, sigma из schedule по индексам t
    # 3. Вернуть x_t = mu * x0 + sigma * eps, eps, mu, sigma

    epsilon = torch.randn(size=x0.shape)
    mu_t, sigma_t = schedule.get(t)
    x_t = mu_t * x0 + sigma_t*epsilon
    
    return x_t, epsilon, mu_t, sigma_t


# --- Проверка: визуализируем зашумление для 2D данных ---
# Генерируем 500 точек из смеси двух гауссиан в 2D
def sample_toy_data(n: int) -> torch.Tensor:
    """
    Смесь двух гауссиан в R^2:
    0.5 * N([-2, 0], I) + 0.5 * N([2, 0], I)
    """
    gauss_1 = torch.randn(n, 2) + torch.tensor([-2, 0])
    gauss_2 = torch.randn(n, 2) + torch.tensor([2, 0])
    probs = torch.round(torch.rand(n, 1))
    return probs * gauss_1 + (1 - probs) * gauss_2


x0 = sample_toy_data(500)

viz_schedule = NoiseSchedule(T=1000)  # локальное расписание, не зависит от обучения

fig, axes = plt.subplots(1, 5, figsize=(15, 3))
eps = torch.randn_like(x0)  # один шум для всех τ — видно постепенное зашумление
for i, tau in enumerate([0, 100, 300, 600, 999]):
    t_batch = torch.full((len(x0),), tau, dtype=torch.long)
    mu_t, sigma_t = viz_schedule.get(t_batch)
    x_t = mu_t * x0 + sigma_t * eps
    axes[i].scatter(x_t[:, 0], x_t[:, 1], s=5, alpha=0.6)
    axes[i].set_title(f'τ = {tau}')
    axes[i].set_xlim(-6, 6)
    axes[i].set_ylim(-6, 6)
    axes[i].set_aspect('equal')

plt.tight_layout()
plt.show()

---
## Блок 3. DSM-лосс: чему мы учим модель

### Связь деноизера со score (формула Твиди)

**Теорема.** Для маргинала VP-SDE:

$$\nabla_{x_\tau} \log p_\tau(x_\tau) = -\frac{\mathbb{E}[\varepsilon \mid x_\tau]}{\sigma(\tau)}$$

**Следствие.** Если обучить сеть $\hat\varepsilon_\theta(x_\tau, \tau)$ предсказывать $\mathbb{E}[\varepsilon \mid x_\tau]$, то:

$$s_\theta(x_\tau, \tau) = -\frac{\hat\varepsilon_\theta(x_\tau, \tau)}{\sigma(\tau)} \approx \nabla_{x_\tau} \log p_\tau(x_\tau)$$

**Предсказывать шум = предсказывать score** (с точностью до нормировки).

### DSM-лосс

Явный score matching $J_{\mathrm{ESM}}$ требует знания $p_\tau$ — недоступно. Через эквивалентность ESM = DSM + const:

$$\mathcal{L}(\theta) = \mathbb{E}_{\tau,\, x_0,\, \varepsilon}\!\left[\left\|\hat\varepsilon_\theta\!\left(\mu(\tau)\, x_0 + \sigma(\tau)\,\varepsilon,\; \tau\right) - \varepsilon\right\|^2\right]$$

Это **среднеквадратичная ошибка предсказания шума** — стандартный лосс DDPM.

### Выбор уровня шума $\tau$

На каждом шаге обучения $\tau$ сэмплируется равномерно: $\tau \sim \mathrm{Uniform}\{1, \ldots, T\}$.

Это важно: модель обучается **на всех уровнях шума**, иначе score будет известен только при одном $\tau$.

In [ ]:
def dsm_loss(model: nn.Module, x0: torch.Tensor, T:int, schedule: NoiseSchedule) -> torch.Tensor:
    """
    Denoising Score Matching loss.

    Алгоритм:
    1. Сэмплировать t ~ Uniform{0, ..., T-1} для каждого элемента батча
    2. Прогнать прямой процесс: x_t, eps, mu, sigma = forward_process(x0, t, schedule)
    3. Предсказать шум: eps_hat = model(x_t, t)
    4. Вернуть MSE: mean ||eps_hat - eps||^2

    Параметры
    ---------
    model    : nn.Module, принимает (x_t, t) -> eps_hat
    x0       : (B, d) — батч чистых данных
    schedule : NoiseSchedule

    Возвращает
    ----------
    loss : скалярный тензор
    """
    # TODO: реализовать по алгоритму выше

    t = torch.randint(0, T, (x0.shape[0],))
    x_t, eps, mu, sigma = forward_process(x0, t, schedule)
    eps_hat = model(x_t, t)

    return torch.mean((eps-eps_hat)**2)

---
## Блок 4. Архитектура деноизера

### Что должна делать сеть

Входы:
- $x_\tau \in \mathbb{R}^d$ — зашумлённые данные,
- $\tau \in \{0,\ldots,T\}$ — уровень шума.

Выход: $\hat\varepsilon_\theta(x_\tau, \tau) \in \mathbb{R}^d$ — предсказанный шум (той же размерности, что вход).

### Как кодировать $\tau$

Уровень шума кодируется **sinusoidal embedding** (как positional encoding в трансформерах):

$$\mathrm{emb}(\tau)_i = \begin{cases} \sin(\tau / 10000^{2i/d_{\mathrm{emb}}}) & i \text{ чётное} \\ \cos(\tau / 10000^{(2i-1)/d_{\mathrm{emb}}}) & i \text{ нечётное} \end{cases}$$

Это даёт непрерывное представление $\tau$, инвариантное к масштабу.

### Архитектура для игрушечного 2D примера

MLP с residual connections и time embedding:

```
x_τ ∈ R^d ──┐
             ├──► [Linear → SiLU] × L + time_emb injection → Linear → eps_hat ∈ R^d
τ ──► emb ──┘
```

Для реальных данных (изображения, стаканы) здесь стоит UNet или Transformer, но для понимания хватит MLP.

In [ ]:
class SinusoidalEmbedding(nn.Module):
    """
    Sinusoidal positional embedding для временного шага τ.

    Параметры
    ---------
    dim : размерность embedding
    """

    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """
        Параметры
        ---------
        t : (B,) LongTensor — шаги от 0 до T-1

        Возвращает
        ----------
        emb : (B, dim) FloatTensor
        """
        # TODO:
        # 1. half_dim = dim // 2
        # 2. freqs = exp(-log(10000) * arange(half_dim) / (half_dim - 1))
        # 3. args = t.float().unsqueeze(1) * freqs.unsqueeze(0)  — (B, half_dim)
        # 4. emb = cat([sin(args), cos(args)], dim=-1)           — (B, dim)

        half_dim = self.dim // 2
        freqs = torch.exp(-math.log(10_000) * torch.arange(half_dim) / (half_dim - 1))
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)  

        return emb

class SimpleDenoiser(nn.Module):
    """
    MLP-деноизер для низкоразмерных данных.

    Архитектура:
        x_τ → Linear(d, hidden)
        t   → SinusoidalEmbedding(emb_dim) → Linear(emb_dim, hidden)
        sum → [SiLU → Linear(hidden, hidden)] × n_layers
            → Linear(hidden, d)

    Параметры
    ---------
    d        : размерность данных
    hidden   : ширина скрытых слоёв
    n_layers : число скрытых блоков
    emb_dim  : размерность time embedding
    """

    def __init__(self, d: int, hidden: int = 128, n_layers: int = 3, emb_dim: int = 32):
        super().__init__()

        # TODO: определить слои
        # self.time_emb   = SinusoidalEmbedding(emb_dim)
        # self.time_proj  = Linear(emb_dim, hidden)
        # self.input_proj = Linear(d, hidden)
        # self.blocks     = ModuleList([Linear(hidden, hidden) × n_layers])
        # self.output     = Linear(hidden, d)

        self.time_emb = SinusoidalEmbedding(emb_dim)
        self.time_proj = torch.nn.Linear(emb_dim, hidden)
        self.input_proj = torch.nn.Linear(d, hidden)
        self.blocks = torch.nn.ModuleList([torch.nn.Linear(hidden, hidden) for _ in range(n_layers)])
        self.output = torch.nn.Linear(hidden, d)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        Параметры
        ---------
        x : (B, d) — зашумлённые данные
        t : (B,)   — шаги шума

        Возвращает
        ----------
        eps_hat : (B, d) — предсказанный шум
        """
        # TODO:
        # h = input_proj(x) + time_proj(time_emb(t))
        # for block in blocks:
        #     h = SiLU(block(h)) + h   ← residual
        # return output(h)

        h = self.input_proj(x) + self.time_proj(self.time_emb(t))
        for block in self.blocks:
            h = torch.nn.functional.silu(block(h)) + h
        return self.output(h)

---
## Блок 5. Цикл обучения

### Алгоритм обучения DDPM

```
for epoch in range(N_epochs):
    for x0 in dataloader:
        loss = dsm_loss(model, x0, schedule)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
```

### Что отслеживать

- **Train loss**: должна монотонно убывать.
- **Визуализация score**: при $\tau \to 0$ вектор $-\hat\varepsilon_\theta(x, 1)/\sigma(1)$ должен указывать к модам распределения.
- **Качество генерации** (опционально): прогнать обратный процесс и сравнить с обучающей выборкой.

In [ ]:
# --- Гиперпараметры ---
D        = 2       # размерность данных (2D игрушка)
N_TRAIN  = 5000    # число обучающих точек
BATCH_SIZE    = 256
LR       = 3e-4
N_EPOCHS = 200
T=200

# --- Данные ---
x_train = sample_toy_data(N_TRAIN).to(device)

# --- Модель и оптимизатор ---
model     = SimpleDenoiser(d=D, hidden=128, n_layers=4).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
schedule = NoiseSchedule(T, 0.1, 0.9)

# --- Цикл обучения ---
losses = []

dataloader = DataLoader(x_train, batch_size=BATCH_SIZE, shuffle=True)
for epoch in range(N_EPOCHS):
    epoch_losses = []
    for batch in dataloader:
        optimizer.zero_grad()
        loss = dsm_loss(model, batch, T, schedule)
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())
    losses.append(sum(epoch_losses) / len(epoch_losses))

# plt.figure(figsize=(6, 3))
# TODO: нарисовать кривую обучения
# plt.xlabel('epoch')
# plt.ylabel('DSM loss')
# plt.title('Training curve')
# plt.tight_layout()
# plt.show()

---
## Блок 5б. Обратный процесс: генерация из шума

### Алгоритм DDPM (сэмплирование)

Запускаем **обратный марковский процесс**, применяя обученный деноизер:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\!\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\hat{\varepsilon}_\theta(x_t, t)\right) + \sqrt{\beta_t}\,z, \quad z \sim \mathcal{N}(0,I)$$

Начинаем с $x_T \sim \mathcal{N}(0,I)$ и повторяем $T$ шагов назад.

In [ ]:
@torch.no_grad()
def ddpm_sample(model, schedule, n_samples=1000, device='cpu'):
    """Обратный процесс DDPM: x_T ~ N(0,I) → x_0 ~ p_data."""
    model.eval()
    T = schedule.T
    x = torch.randn(n_samples, 2, device=device)

    save_steps = {T - 1, 3 * T // 4, T // 2, T // 4, 0}
    snapshots = {}

    for t in reversed(range(T)):
        t_batch = torch.full((n_samples,), t, dtype=torch.long, device=device)
        eps_hat = model(x, t_batch)

        beta_t  = schedule.betas[t].to(device)
        alpha_t = schedule.alphas[t].to(device)
        sigma_t = schedule.sigmas[t].to(device)  # sqrt(1 - alpha_bar_t)

        # x_{t-1} = (x_t - beta_t / sigma_t * eps_hat) / sqrt(alpha_t)  +  sqrt(beta_t) * z
        x = (x - beta_t / sigma_t * eps_hat) / alpha_t.sqrt()
        if t > 0:
            x = x + beta_t.sqrt() * torch.randn_like(x)

        if t in save_steps:
            snapshots[t] = x.detach().cpu()

    return snapshots


snapshots = ddpm_sample(model, schedule, n_samples=1000, device=device)

steps = sorted(snapshots.keys(), reverse=True)  # T-1 → 0
fig, axes = plt.subplots(1, len(steps), figsize=(15, 3))
for ax, t in zip(axes, steps):
    pts = snapshots[t]
    ax.scatter(pts[:, 0], pts[:, 1], s=4, alpha=0.5)
    ax.set_title(f't = {t}')
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_aspect('equal')

# Для сравнения: реальные данные
x_real = sample_toy_data(1000)
fig2, ax2 = plt.subplots(figsize=(3.5, 3.5))
ax2.scatter(x_real[:, 0], x_real[:, 1], s=4, alpha=0.5, color='green')
ax2.set_title('Реальные данные')
ax2.set_xlim(-5, 5); ax2.set_ylim(-5, 5); ax2.set_aspect('equal')

plt.tight_layout()
plt.show()